# Python to C++

Each Azure deployment ports the same pi loop to C++. The C++ is saved as `main.cpp` in this folder, next to the notebook, then compiled with `g++`. The Python run is the baseline.


## Client

In [1]:
import os
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
day_dir = repo_root / "W4-Model Evaluation" / "Day-3"
sys.path.insert(0, str(day_dir))
load_dotenv(repo_root / ".env", override=True)

from system_info import retrieve_system_info

client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
azure_model = os.getenv("GPT_MODEL", "gpt-4o")
deployments = []
for name in (azure_model, "gpt-4o-mini", "gpt-4.1"):
    if name not in deployments:
        deployments.append(name)

system_info = retrieve_system_info()
cpp_path = day_dir / "main.cpp"
binary_path = day_dir / "main"
compile_command = [
    "g++",
    "-std=c++17",
    "-O3",
    "-march=native",
    str(cpp_path),
    "-o",
    str(binary_path),
]
run_command = [str(binary_path)]
print(cpp_path)
print(compile_command)


/home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/W4-Model Evaluation/Day-3/main.cpp
['g++', '-std=c++17', '-O3', '-march=native', '/home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/W4-Model Evaluation/Day-3/main.cpp', '-o', '/home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/W4-Model Evaluation/Day-3/main']


## Port

In [2]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""


def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""


def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)},
    ]


def cpp_only(text):
    text = (text or "").strip()
    if "```" not in text:
        return text + "\n"
    block = text.split("```", 2)[1]
    if block.startswith("cpp") or block.startswith("c++"):
        block = block.split("\n", 1)[-1]
    return block.strip() + "\n"


def port(deployment, python):
    response = client.chat.completions.create(
        model=deployment,
        messages=messages_for(python),
        timeout=120.0,
    )
    cpp = cpp_only(response.choices[0].message.content)
    cpp_path.write_text(cpp, encoding="utf-8")
    print(cpp_path)
    return cpp


def compile_and_run():
    if not compile_command:
        print("No C++ compiler on PATH.")
        print("https://www.programiz.com/cpp-programming/online-compiler/")
        return ""
    try:
        subprocess.run(
            compile_command,
            check=True,
            text=True,
            capture_output=True,
            timeout=120,
        )
        result = subprocess.run(
            run_command,
            check=True,
            text=True,
            capture_output=True,
            timeout=60,
        )
    except subprocess.CalledProcessError as exc:
        print(exc.stderr or exc)
        return ""
    print(result.stdout)
    return result.stdout


## Python baseline

Two hundred million iterations. This cell is the slow number the C++ runs get compared with.

In [3]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""


def run_python(code):
    exec(code, {"__builtins__": __builtins__})


run_python(pi)


Result: 3.141592656089
Execution Time: 25.871956 seconds


## Three deployments

Each reply overwrites `main.cpp` next to this notebook, then compiles it.

In [4]:
for deployment in deployments:
    print(f"=== {deployment} ===")
    port(deployment, pi)
    compile_and_run()


=== gpt-4o ===
/home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/W4-Model Evaluation/Day-3/main.cpp
Result: 3.141592656089
Execution Time: 0.480724 seconds

=== gpt-4o-mini ===
/home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/W4-Model Evaluation/Day-3/main.cpp
Result: 3.141592656089
Execution Time: 0.481894381000 seconds

=== gpt-4.1 ===
/home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/W4-Model Evaluation/Day-3/main.cpp
Result: 3.141592656089
Execution Time: 0.472084 seconds

